In [ ]:
#| default_exp spec

In [ ]:
#| hide
from fastcore.test import *
from nbdev.showdoc import *
from tempfile import TemporaryDirectory
from fastcore.all import Path

What an app is made of, said once, for whichever freezer this platform uses.

An `App` is a description, and describing one builds nothing. This module imports nothing but
fastcore and the standard library, so a machine that cannot build a Mac app can still say what
building one would take.

`py2app_options` and `py2exe_options` read the same fields. The platform decides which one a caller
asks for, never what the app is.

In [ ]:
#| export
from __future__ import annotations
import sys
from dataclasses import dataclass, field
from fastcore.all import Path, globtastic, groupby

In [ ]:
#| export
def tree(src, dest, skip=()):
    "Every file under `src` as freezer `(dest_dir, [files])` pairs, recursively."
    fs = globtastic(src, skip_folder_re=f'^({"|".join([*skip, "__pycache__"])})$', skip_file_re=r'^\.')
    dirs = groupby(fs, lambda f: str(Path(dest)/Path(f).parent.relative_to(src)))
    return sorted((d, sorted(v)) for d, v in dirs.items())

def mypyc_modules(site=None):
    """Compiled accelerators that sit beside a package rather than inside it.

    `chardet` is built with mypyc, so importing it imports a top-level extension whose name is a
    hash of the build. Nothing references it by name, so no scanner finds it, and the package looks
    complete without it. The hash moves with the version, so it is read from the environment being
    frozen rather than written down.
    """
    import sysconfig
    site = Path(site or sysconfig.get_paths()['purelib'])
    return (sorted(f.name.split('.')[0] for f in site.glob('*__mypyc*.so')) +
            sorted(f.name.split('.')[0] for f in site.glob('*__mypyc*.pyd')))

`tree` turns a directory into the `(dest_dir, [files])` pairs both freezers take as `data_files`.
The pairs and the files inside each pair come back sorted, so the same tree gives the same list on
every machine and a build is comparable with the one before it.

`__pycache__` and dotfiles are dropped. `skip` names further folders, matched on the folder name at
any depth. A directory with no files in it gets no pair: a freezer copies files, and there is
nothing there to copy.

The destinations are the paths inside the bundle. The files are the paths on this disk, left as
`globtastic` found them, because that is what the freezer has to read.

In [ ]:
#| hide
tmp = TemporaryDirectory(); src = Path(tmp.name)/'static'
for d in ('js', 'img', 'node_modules', 'empty'): (src/d).mkdir(parents=True)
(src/'app.css').write_text('a{}'); (src/'.DS_Store').write_text('')
(src/'js'/'app.js').write_text('//'); (src/'img'/'logo.png').write_text('x')
(src/'node_modules'/'left-pad.js').write_text('//')

2

In [ ]:
data = tree(src, 'demo/static', skip=['node_modules'])
[(dest, [Path(f).name for f in files]) for dest, files in data]

[('demo/static', ['app.css']),
 ('demo/static/img', ['logo.png']),
 ('demo/static/js', ['app.js'])]

In [ ]:
#| hide
test_eq([dest for dest, _ in data], ['demo/static', 'demo/static/img', 'demo/static/js'])
assert all(Path(f).is_file() for _, files in data for f in files), 'real paths, for the freezer to read'

`mypyc_modules` reads the environment being frozen rather than a written-down list, because the
hash in the name moves with the version. It looks beside the packages in `site`, which defaults to
this interpreter's `purelib`, and it matches on the shape of the filename. A module whose name has
no `__mypyc` in it is somebody else's extension and is left alone.

In [ ]:
site = Path(tmp.name)/'site'; site.mkdir()
for f in ('chardet__mypyc.cpython-313-darwin.so', 'numpy.cpython-313-darwin.so'): (site/f).write_text('')
mypyc_modules(site)

['chardet__mypyc']

In [ ]:
#| export
#: Build tooling and the GUI toolkits no platform's webview uses: about 100MB a frozen app never
#: wanted. Nothing here belongs to one platform; `GUI_MODULE` and `GUI_SUPPORT` hold those.
DEFAULT_EXCLUDES = ('tkinter', 'test', 'setuptools', 'wheel', 'py2app', 'py2exe',
                    'PyQt5', 'PyQt6', 'PySide2', 'PySide6', 'matplotlib')

#: The webview backends each platform does want, so the others can be excluded.
GUI_MODULE = {'darwin': 'webview.platforms.cocoa', 'win32': 'webview.platforms.winforms',
              'linux': 'webview.platforms.gtk'}

#: What each backend imports besides itself. Excluding one platform's set on another is safe;
#: excluding it on its own platform takes the window with it, which is what `clr` in
#: `DEFAULT_EXCLUDES` did to every py2exe build: pythonnet provides it, and `winforms` needs it.
GUI_SUPPORT = {'darwin': ('objc', 'Foundation', 'AppKit', 'WebKit', 'Quartz'),
               'win32': ('clr', 'clr_loader', 'pythonnet'),
               'linux': ('gi',)}

`DEFAULT_EXCLUDES` is what a frozen app never wanted and would otherwise carry: the build tooling
that froze it, the standard library's test suite, and the GUI toolkits that arrive as somebody
else's dependency. About 100MB. Nothing in it belongs to one platform.

`GUI_MODULE` names the one webview backend each platform loads, and `GUI_SUPPORT` what that backend
imports besides itself. `App.excluded` leaves out the other two platforms' entries from both, so
what ships is the backend the app can open a window with and nothing else.

In [ ]:
#| export
@dataclass
class App:
    """One desktop app, described once. `build` turns this into whichever freezer fits the platform.

    `packages` is the part nobody can guess: a freezer's scanner cannot see an import made inside a
    function, which is most of any real application, so packages are named rather than discovered.
    `includes` is the same for single modules. Both are additive to what the scanner does find.
    """
    name: str
    entry: str                                  # the script the bundle runs
    version: str = '0.0.0'
    icon: str = ''                              # an .icns/.ico; `kavacha.icons` makes one
    modern_icon: str = ''                       # a macOS 26 Icon Composer document
    identifier: str = ''                        # reverse-DNS bundle id; derived from `name` if unset
    summary: str = ''
    packages: list = field(default_factory=list)    # copied whole, not scanned
    includes: list = field(default_factory=list)    # single modules the scanner cannot reach
    excludes: list = field(default_factory=list)    # on top of `DEFAULT_EXCLUDES`
    grafted: list = field(default_factory=list)     # put in place after the freezer runs
    unreachable: list = field(default_factory=list) # archive prefixes nothing can read
    data: list = field(default_factory=list)        # `(dest_dir, [files])` pairs, as `tree` makes
    plist: dict = field(default_factory=dict)       # merged over the defaults below
    doc_types: list = field(default_factory=list)
    extras: list = field(default_factory=list)      # the extras a build environment installs
    min_macos: str = '11.0'
    category: str = 'public.app-category.developer-tools'

    def __post_init__(self):
        self.identifier = self.identifier or f'org.{self.name.lower()}.{self.name.lower()}'
    @property
    def bundle_name(self): return f'{self.name}.app' if sys.platform == 'darwin' else self.name
    def out(self, root): return Path(root)/'dist'/self.bundle_name
    def info_plist(self):
        "The `Info.plist` a macOS bundle needs, with this app's entries over the ones every app wants."
        base = {
            'CFBundleName': self.name, 'CFBundleDisplayName': self.name,
            'CFBundleIdentifier': self.identifier,
            'CFBundleVersion': self.version, 'CFBundleShortVersionString': self.version,
            'CFBundleGetInfoString': self.summary or self.name,
            'LSApplicationCategoryType': self.category,
            'LSMinimumSystemVersion': self.min_macos,
            'NSHighResolutionCapable': True,
            # Without this the app is locked to the light appearance, and a dark web UI renders
            # against a white window.
            'NSRequiresAquaSystemAppearance': False,
            # App Transport Security blocks cleartext HTTP, loopback included, so WKWebView refuses
            # `http://127.0.0.1:<port>`. The narrow exemption, not `NSAllowsArbitraryLoads`.
            'NSAppTransportSecurity': {'NSAllowsLocalNetworking': True},
            # Set before the interpreter starts, which nothing in Python can be. Both are wanted.
            'LSEnvironment': {'LANG': 'en_US.UTF-8', 'PYTHONUTF8': '1'},
        }
        if self.doc_types: base['CFBundleDocumentTypes'] = self.doc_types
        return base | dict(self.plist)

    def excluded(self, platform=None):
        "Everything this build leaves out, including the webview backends it is not using."
        mine = platform or sys.platform
        others = [GUI_MODULE[p] for p in GUI_MODULE if p != mine]
        others += [m for p, ms in GUI_SUPPORT.items() if p != mine for m in ms]
        return sorted({*DEFAULT_EXCLUDES, *others, *self.excludes})

    def py2app_options(self):
        "The `options={'py2app': ...}` dict, built from this spec."
        out = {'packages': list(self.packages), 'includes': list(self.includes),
               'excludes': self.excluded('darwin'), 'plist': self.info_plist(),
               # Opened from the Dock and from `open`, never with a shell's argv conventions;
               # argv emulation costs a visible Apple Event wait at every launch.
               'argv_emulation': False, 'semi_standalone': False, 'site_packages': True,
               'strip': True}
        if self.icon: out['iconfile'] = str(self.icon)
        return out

    def py2exe_options(self):
        "The `options={'py2exe': ...}` dict, built from this spec."
        return {'packages': list(self.packages), 'includes': list(self.includes),
                'excludes': self.excluded('win32'), 'bundle_files': 3, 'compressed': 1}

`identifier` is derived from `name` when the caller gives none, as `org.<name>.<name>` lowercased.
An app that will be signed or notarized wants a real one.

`bundle_name` and `out` follow `sys.platform`, the machine describing the app, not the platform the
app is for. Describing a Mac app on Linux gives `Demo`, not `Demo.app`. Both are answers about
where this machine would put a build, and only the platform that builds is asked.

`info_plist` merges the caller's `plist` over the defaults, so any of them can be overridden,
including the three an app is silently broken without. `excluded` merges the same way and stays
sorted and deduplicated.

The two options dicts read `packages`, `includes`, `excludes`, `plist` and `icon`, and copy every
list on the way out. `data`, `grafted`, `unreachable`, `extras`, `modern_icon` and `version` appear
in neither: they are read by `build`, `finish` and the bundle surgery, after the freezer has run.

In [ ]:
app = App(name='Demo', entry='demo_app.py', version='1.2.3', summary='the demo app',
          icon='assets/Demo.icns', packages=['demo', 'fasthtml', 'uvicorn'],
          includes=['demo.cli'], data=data, extras=['desktop'])
app.identifier, app.bundle_name, str(app.out('/build/demo'))

('org.demo.demo', 'Demo', '/build/demo/dist/Demo')

In [ ]:
app.info_plist()

{'CFBundleName': 'Demo',
 'CFBundleDisplayName': 'Demo',
 'CFBundleIdentifier': 'org.demo.demo',
 'CFBundleVersion': '1.2.3',
 'CFBundleShortVersionString': '1.2.3',
 'CFBundleGetInfoString': 'the demo app',
 'LSApplicationCategoryType': 'public.app-category.developer-tools',
 'LSMinimumSystemVersion': '11.0',
 'NSHighResolutionCapable': True,
 'NSRequiresAquaSystemAppearance': False,
 'NSAppTransportSecurity': {'NSAllowsLocalNetworking': True},
 'LSEnvironment': {'LANG': 'en_US.UTF-8', 'PYTHONUTF8': '1'}}

The three entries nothing else sets. `NSAppTransportSecurity` is the narrow local-networking
exemption rather than `NSAllowsArbitraryLoads`: without it WKWebView refuses `http://127.0.0.1`.
`NSRequiresAquaSystemAppearance` is False, or a dark web UI renders against a white window.
`PYTHONUTF8` is in `LSEnvironment` because it has to be set before the interpreter starts, which
nothing written in Python can do.

In [ ]:
app.py2app_options()['excludes']

['PyQt5',
 'PyQt6',
 'PySide2',
 'PySide6',
 'clr',
 'gi',
 'matplotlib',
 'py2app',
 'py2exe',
 'setuptools',
 'test',
 'tkinter',
 'webview.platforms.gtk',
 'webview.platforms.winforms',
 'wheel']

The two backends this build is not using are in that list, and the one it is using is not.

In [ ]:
o = app.py2app_options()
o['packages'].append('scipy')
test_eq(app.packages, ['demo', 'fasthtml', 'uvicorn'])   # the options are a copy, not the spec
noisy = App(name='D', entry='e.py', excludes=['mlx', 'mlx', 'tkinter']).excluded('darwin')
test_eq(noisy.count('tkinter'), 1)
test_eq(noisy, sorted(set(noisy)))

In [ ]:
#| export
def doc_types(extensions, view_elsewhere=('.svg', '.html', '.htm', '.pdf')):
    """Finder document types for an editor: one entry for folders, one for the files it owns.

    A folder stays `Alternate` — Finder is the right default for one, and this is what puts
    "Open With" on it and makes a Dock drop mean something. `view_elsewhere` are the extensions a
    browser renders and an editor would only show the source of.
    """
    exts = sorted({str(e).lstrip('.') for e in extensions if e not in view_elsewhere})
    return [
        {'CFBundleTypeName': 'Folder', 'CFBundleTypeRole': 'Viewer',
         'LSItemContentTypes': ['public.folder'], 'LSHandlerRank': 'Alternate'},
        {'CFBundleTypeName': 'Source file', 'CFBundleTypeRole': 'Editor',
         'CFBundleTypeExtensions': exts, 'LSHandlerRank': 'Owner'},
    ]

`doc_types` returns two entries whatever the extension list is: one for folders, one for the files
the app owns. The folder entry stays `Alternate`, so Finder keeps opening folders and the app still
appears under "Open With" and accepts a Dock drop. The file entry is `Owner`.

Extensions come back without the leading dot, sorted, once each. `view_elsewhere` is compared
against the values as given, so pass extensions with the dot for the default to apply.

In [ ]:
folder, files = doc_types(['.py', 'py', '.rs', '.svg', '.pdf'])
files['CFBundleTypeExtensions'], folder['LSHandlerRank'], files['LSHandlerRank']

(['py', 'rs'], 'Alternate', 'Owner')

In [ ]:
test_eq(files['CFBundleTypeExtensions'], ['py', 'rs'])   # dotted or not, once each, and no .svg
test_eq(folder['LSItemContentTypes'], ['public.folder'])

In [ ]:
#| hide
tmp.cleanup()